# Train L2 (CrossEntityAttention) — uniform T=10 history window

Pretrains the L2 cross-entity self-attention layer plus the V2 PlayerConsolidator-aware player/global head menu on the cross_entity dataset. L0 (PlanetEncoder, FleetEncoder) and L1 (PlanetEntityEncoder) are loaded from prior runs and frozen.

**History window:** `HISTORY_OFFSETS = (45, 40, 35, 30, 25, 20, 15, 10, 5, 0)` — uniform 5-turn spacing, 10 slots, ~50-turn lookback. `step_embed` sized to n_steps=10.

**Data path: sharded CSV-walk.** Pulls/extracts `cross_entity_dataset_shard_00.tgz` ... `cross_entity_dataset_shard_07.tgz` in parallel by default, then lets `CrossEntitySnapshotDataset` walk `data/datasets/{planet,fleet,entity,cross_entity}/`. Code/weights pulls and extraction run in parallel with dataset staging, and CSV parsing uses separate parallel load workers before training. The trainer reads `manifest.json` for the 80/10/10 train/val/test split. The `.pt` cache path is intentionally not used here — large GCS object pulls were the bottleneck. Set `DATASET_SHARDS = 0` below only to fall back to streaming the legacy single object `cross_entity_dataset.tgz`.

## Inputs from GCS

```
gs://orbit-wars-shipping/cross_entity/
  code.tgz                          # agents/ + scripts/
  weights.tgz                       # frozen L0 (planet, fleet, comet) d=256
  cross_entity_dataset_shard_00.tgz # CSV shard archives, 00..07 by default
  ...
  cross_entity_dataset_shard_07.tgz
  cross_entity_dataset_shard.manifest.json
  cross_entity_dataset.tgz          # legacy single-object fallback
  entity_encoder_best.pt            # frozen L1 (May-21 baseline)
  manifest.json                     # split lists
```


## 0. Config

In [ ]:
BATCH_SIZE  = 256
EPOCHS      = 10
LR          = 1e-3
NUM_WORKERS = 2
NUM_LOAD_WORKERS = 8

# Default: pull/extract independent shard archives in parallel:
#   cross_entity_dataset_shard_00.tgz, ...
# Set to 0 to stream the legacy single cross_entity_dataset.tgz object.
DATASET_SHARDS = 8
DATASET_SHARD_PREFIX = 'cross_entity_dataset_shard'
print(f'batch={BATCH_SIZE}  epochs={EPOCHS}  lr={LR}  workers={NUM_WORKERS}  load_workers={NUM_LOAD_WORKERS}  shards={DATASET_SHARDS}')

## 1. Authenticate + pull bundle

In [ ]:
from google.colab import auth
auth.authenticate_user()
BUCKET = 'gs://orbit-wars-shipping/cross_entity'
print(f'pulling from {BUCKET}')

In [ ]:
import os, shutil, subprocess, time, concurrent.futures
from pathlib import Path

WORK = Path('/content/orbit-wars')
WORK.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)

# Wipe stale extracted state before parallel staging starts.
for rel in ('agents', 'scripts', 'ckpts', 'data/datasets', 'data/runs'):
    shutil.rmtree(WORK / rel, ignore_errors=True)
(WORK / 'data/datasets').mkdir(parents=True, exist_ok=True)
for rel in ('cross_entity', 'entity', 'fleet', 'planet'):
    (WORK / 'data/datasets' / rel).mkdir(parents=True, exist_ok=True)

def cp(src, dst, *, force=True):
    dst = Path(dst)
    if dst.exists() and not force:
        return dst.name, 0.0, dst.stat().st_size
    if dst.exists():
        dst.unlink()
    t0 = time.time()
    print(f'  pulling {src} → {dst.name} ...', flush=True)
    subprocess.run(['gcloud', 'storage', 'cp', src, str(dst)], check=True)
    return dst.name, time.time() - t0, dst.stat().st_size

def stream_dataset(src, name='cross_entity_dataset.tgz'):
    t0 = time.time()
    print(f'  streaming {src} → tar -C data/datasets ...', flush=True)
    cat = subprocess.Popen(['gcloud', 'storage', 'cat', src], stdout=subprocess.PIPE)
    try:
        subprocess.run(['tar', '-C', 'data/datasets', '-xzf', '-'], stdin=cat.stdout, check=True)
    finally:
        if cat.stdout is not None:
            cat.stdout.close()
    rc = cat.wait()
    if rc != 0:
        raise subprocess.CalledProcessError(rc, ['gcloud', 'storage', 'cat', src])
    return name, time.time() - t0, -1, 'dataset-stream', 0.0

def pull_and_stage(src, dst, stage):
    if stage == 'dataset':
        return stream_dataset(src, Path(src).name)
    name, dt, size = cp(src, dst)
    t0 = time.time()
    if stage == 'code':
        subprocess.run(['tar', 'xzf', str(dst)], check=True)
    elif stage == 'weights':
        subprocess.run(['tar', 'xzf', str(dst)], check=True)
    return name, dt, size, stage, time.time() - t0

TASKS = [
    (f'{BUCKET}/code.tgz',                  WORK / 'code.tgz',                 'code'),
    (f'{BUCKET}/weights.tgz',               WORK / 'weights.tgz',              'weights'),
    (f'{BUCKET}/entity_encoder_best.pt',    WORK / 'entity_encoder_best.pt',   'file'),
    (f'{BUCKET}/manifest.json',             WORK / 'manifest.json',            'file'),
]
if DATASET_SHARDS:
    TASKS.extend(
        (f'{BUCKET}/{DATASET_SHARD_PREFIX}_{i:02d}.tgz', None, 'dataset')
        for i in range(DATASET_SHARDS)
    )
else:
    TASKS.append((f'{BUCKET}/cross_entity_dataset.tgz', None, 'dataset'))
T_START = time.time()
results = []
with concurrent.futures.ThreadPoolExecutor(max_workers=len(TASKS)) as pool:
    futs = [pool.submit(pull_and_stage, s, d, stage) for s, d, stage in TASKS]
    for f in concurrent.futures.as_completed(futs):
        results.append(f.result())
for name, dt, size, stage, extract_dt in sorted(results, key=lambda t: -t[2]):
    size_msg = ' streamed' if size < 0 else f'{size / 1024 / 1024:>9.1f} MB'
    print(f'  {name:<34s} {size_msg}  pull={dt:5.1f}s  stage={extract_dt:5.1f}s  {stage}')
# Manifest lives next to the cross_entity CSVs.
(WORK / 'data/datasets/cross_entity').mkdir(parents=True, exist_ok=True)
shutil.copy(WORK / 'manifest.json', WORK / 'data/datasets/cross_entity/manifest.json')
print(f'\ntotal wall: {time.time()-T_START:.1f}s')

In [ ]:
# Parallel staging happened in the previous cell. This cell only clears
# stale imports and prints a quick filesystem sanity check.
import sys
for m in list(sys.modules):
    if m.startswith('agents') or m.startswith('scripts'):
        del sys.modules[m]
import importlib, gc
importlib.invalidate_caches()
gc.collect()
!find . -type d -name __pycache__ -exec rm -rf {} + 2>/dev/null || true
!du -sh data/datasets/*
!ls -la

## 1b. Verify HISTORY_OFFSETS = T=10

In [ ]:
import agents
from agents.transformer_v2.history import HISTORY_OFFSETS, N_HISTORY
print(f'agents module: {agents.__file__}')
print(f'HISTORY_OFFSETS: {HISTORY_OFFSETS}')
assert HISTORY_OFFSETS == (45, 40, 35, 30, 25, 20, 15, 10, 5, 0)
assert N_HISTORY == 10

from agents.transformer_v2.pretrain.cross_entity import CrossEntityPretrainModelV2
import torch
m = CrossEntityPretrainModelV2(d_model=256)
assert m.cross.step_embed.shape == (10, 256), m.cross.step_embed.shape
assert hasattr(m, 'consolidator')
print(f'L2 step_embed table: {tuple(m.cross.step_embed.shape)} (T=10 confirmed)')

## 2. Verify GPU

In [ ]:
import torch
print(f'torch: {torch.__version__}, cuda available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'  device: {torch.cuda.get_device_name(0)}')
    print(f'  mem total: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

## 3. Stage L0 + L1 ckpts

In [ ]:
import shutil
from pathlib import Path
PLANET_RUN_DIR = Path('/content/orbit-wars/ckpts/planet')
FLEET_RUN_DIR  = Path('/content/orbit-wars/ckpts/fleet')
ENTITY_RUN_DIR = Path('/content/orbit-wars/ckpts/entity')
for d in (PLANET_RUN_DIR, FLEET_RUN_DIR, ENTITY_RUN_DIR):
    d.mkdir(parents=True, exist_ok=True)
shutil.copy('/content/orbit-wars/planet_encoder_best.pt', PLANET_RUN_DIR / 'planet_encoder_best.pt')
shutil.copy('/content/orbit-wars/fleet_encoder_best.pt',  FLEET_RUN_DIR  / 'fleet_encoder_best.pt')
shutil.copy('/content/orbit-wars/entity_encoder_best.pt', ENTITY_RUN_DIR / 'entity_encoder_best.pt')

import torch
for tag, p in (('planet', PLANET_RUN_DIR / 'planet_encoder_best.pt'),
                ('fleet',  FLEET_RUN_DIR  / 'fleet_encoder_best.pt'),
                ('entity', ENTITY_RUN_DIR / 'entity_encoder_best.pt')):
    c = torch.load(p, map_location='cpu', weights_only=False)
    print(f'{tag:6s} ckpt: d_model={c["config"]["d_model"]}, epoch={c["epoch"]}, '
          f'use_traj_branch={c["config"].get("use_traj_branch")}')
    assert c['config']['d_model'] == 256

## 4. Train L2

No `--cross-cache-path` — runs `CrossEntitySnapshotDataset` directly on the extracted CSVs. CSV parsing uses parallel load workers once at startup; subsequent iterations are pure in-memory tensor stacking.

In [ ]:
D_MODEL    = 256
WEIGHT_DECAY = 1e-4
SEED       = 1729
DEVICE     = 'cuda' if __import__('torch').cuda.is_available() else 'cpu'

import time
TS = time.strftime('%Y%m%d-%H%M%S')
RUN_TAG = f'cross_T10_v2_playerpair_d{D_MODEL}_b{BATCH_SIZE}_{EPOCHS}ep_lr{LR:g}_{TS}'
OUT_DIR = f'data/runs/cross_entity/{RUN_TAG}'
print('out dir:', OUT_DIR)

In [ ]:
!python -u -m agents.transformer_v2.pretrain.cross_entity \
  --train-mode frozen \
  --fleet-run-dir  $FLEET_RUN_DIR \
  --planet-run-dir $PLANET_RUN_DIR \
  --entity-run-dir $ENTITY_RUN_DIR \
  --out-dir $OUT_DIR \
  --d-model $D_MODEL \
  --batch-size $BATCH_SIZE \
  --epochs $EPOCHS \
  --lr $LR \
  --weight-decay $WEIGHT_DECAY \
  --head-set v2 \
  --seed $SEED \
  --num-load-workers $NUM_LOAD_WORKERS \
  --num-workers $NUM_WORKERS \
  --device $DEVICE

## 5. Push the trained run back to GCS

In [ ]:
import subprocess
from pathlib import Path
src = Path(OUT_DIR)
assert src.is_dir(), src
dst_parent = f'{BUCKET}/runs/'
subprocess.run(['gcloud', 'storage', 'cp', '--recursive', str(src), dst_parent], check=True)
print(f'uploaded to: {dst_parent}{src.name}/')
subprocess.run(['gcloud', 'storage', 'ls', '--long', '--readable-sizes', f'{dst_parent}{src.name}/'], check=False)